# Chapter 32: Multiview Geometry

<a href="../lite/lab/index.html?path=ch32_multiview_geometry.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Take two photos of the same scene from different positions. Each photo alone is flat,
with no depth. But together, they contain enough information to reconstruct the 3D world.
The mathematical key is the **epipolar constraint**: a point in image 1 constrains where
its match can appear in image 2 to a single line.

```{admonition} What you will build
:class: tip

- Draw epipolar lines and verify that corresponding points lie on them
- Estimate the essential matrix using the 8 point algorithm
- Triangulate 3D points from two camera views
- Recover the relative camera pose (R, t) from the essential matrix

**Real world application:** Two view geometry is the foundation of visual odometry and visual SLAM initialization. After this chapter, you can reconstruct 3D structure from two photographs.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **OpenCV findEssentialMat** | Essential matrix estimation with RANSAC |
| **OpenCV recoverPose** | Decompose essential matrix into R, t |
| **OpenCV triangulatePoints** | DLT triangulation from two views |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 32.1 Epipolar Geometry

Consider two cameras observing the same 3D point $\mathbf{P}$. The point, the two camera
centres, and the two image points all lie in a single plane called the **epipolar plane**.

The intersection of this plane with each image defines an **epipolar line**. A point
$\mathbf{p}_1$ in image 1 constrains its correspondence in image 2 to lie on the
epipolar line $\mathbf{l}_2 = F\,\mathbf{p}_1$, where $F$ is the **fundamental matrix**.

In normalised coordinates, the **essential matrix** $E$ satisfies:

$$\hat{\mathbf{p}}_2^T\, E\, \hat{\mathbf{p}}_1 = 0$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_points  = 30
baseline  = 1.0          # camera separation in metres (try 0.5, 1.0, 2.0)
depth_lo  = 3.0
depth_hi  = 8.0
fx, fy    = 500, 500
cx, cy    = 320, 240
img_w, img_h = 640, 480
# ─────────────────────────────────────────────────────────────────────────────

K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]], dtype=float)
K_inv = np.linalg.inv(K)

# 3D scene points
points_3d = np.column_stack([
    np.random.uniform(-3, 3, n_points),
    np.random.uniform(-2, 2, n_points),
    np.random.uniform(depth_lo, depth_hi, n_points)
])

# Camera 1: identity pose at origin
R1 = np.eye(3); t1 = np.zeros(3)
# Camera 2: translated along X axis
R2 = np.eye(3); t2 = np.array([baseline, 0.0, 0.0])

def project(K, R, t, pts):
    """Project world points to pixel coordinates."""
    pts_cam = (R @ (pts - t).T).T
    proj = (K @ pts_cam.T).T
    pix = proj[:, :2] / proj[:, 2:3]
    return pix, pts_cam[:, 2]

pix1, _ = project(K, R1, t1, points_3d)
pix2, _ = project(K, R2, t2, points_3d)

# Essential matrix: E = [t]_x R  (relative: R2 R1^T = I, t = t2 - t1)
t_rel = t2 - t1
tx = np.array([[ 0,        -t_rel[2],  t_rel[1]],
               [ t_rel[2],  0,        -t_rel[0]],
               [-t_rel[1],  t_rel[0],  0       ]])
E_true = tx @ R2  # R_rel = I here

# Fundamental matrix: F = K^{-T} E K^{-1}
F_true = K_inv.T @ E_true @ K_inv

In [ ]:
# Visualise epipolar lines for 5 selected points
sel = [0, 5, 10, 15, 20]
colors_sel = ['steelblue', 'tomato', 'orange', 'forestgreen', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
ax1.set_title('Image 1: selected points', fontsize=13)
ax1.scatter(pix1[:, 0], pix1[:, 1], c='gray', s=15, alpha=0.3)
for idx, col in zip(sel, colors_sel):
    ax1.scatter(pix1[idx, 0], pix1[idx, 1], c=col, s=80, zorder=5, edgecolors='k')
    ax1.annotate(f'{idx}', pix1[idx] + [5, -8], fontsize=9, color=col, fontweight='bold')
ax1.set_xlim(0, img_w); ax1.set_ylim(img_h, 0)
ax1.set_xlabel('u'); ax1.set_ylabel('v'); ax1.set_aspect('equal')

ax2 = axes[1]
ax2.set_title('Image 2: epipolar lines + true correspondences', fontsize=13)
ax2.scatter(pix2[:, 0], pix2[:, 1], c='gray', s=15, alpha=0.3)

for idx, col in zip(sel, colors_sel):
    # Epipolar line in image 2: l2 = F p1
    p1h = np.array([pix1[idx, 0], pix1[idx, 1], 1.0])
    l2 = F_true @ p1h  # line [a, b, c]: ax + by + c = 0
    # Draw line across image
    u_range = np.array([0, img_w])
    v_range = -(l2[0] * u_range + l2[2]) / l2[1]
    ax2.plot(u_range, v_range, color=col, lw=1.5, alpha=0.7)
    # True correspondence
    ax2.scatter(pix2[idx, 0], pix2[idx, 1], c=col, s=80, zorder=5, edgecolors='k')
    # Verify: distance from true point to epipolar line
    p2h = np.array([pix2[idx, 0], pix2[idx, 1], 1.0])
    dist = abs(p2h @ l2) / np.sqrt(l2[0]**2 + l2[1]**2)
    ax2.annotate(f'{idx} (d={dist:.2e})', pix2[idx] + [5, -8],
                 fontsize=8, color=col)

ax2.set_xlim(0, img_w); ax2.set_ylim(img_h, 0)
ax2.set_xlabel('u'); ax2.set_ylabel('v'); ax2.set_aspect('equal')

plt.tight_layout()
plt.show()
print('Each true correspondence lies ON its epipolar line (distance ~ 0).')

**Key observation:** The distances printed above are essentially zero (machine precision),
confirming that the true correspondence satisfies the epipolar constraint $\mathbf{p}_2^T F \mathbf{p}_1 = 0$.

## 32.2 Essential Matrix: The 8 Point Algorithm

Given $N \ge 8$ point correspondences in **normalised** coordinates
$(\hat{\mathbf{p}}_1, \hat{\mathbf{p}}_2)$, we can estimate $E$ by solving a
homogeneous system.

Each correspondence gives one equation:

$$\hat{\mathbf{p}}_2^T E \hat{\mathbf{p}}_1 = 0
\quad \Rightarrow \quad
\begin{bmatrix} x_2 x_1 & x_2 y_1 & x_2 & y_2 x_1 & y_2 y_1 & y_2 & x_1 & y_1 & 1 \end{bmatrix}
\text{vec}(E) = 0$$

We stack $N$ such rows into $A$ and find the null space via SVD. Then we enforce the
rank 2 constraint by zeroing the smallest singular value of the raw $E$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
pixel_noise_8pt = 0.5     # pixel noise std (try 0, 0.5, 2.0, 5.0)
n_corr = 30               # correspondences to use (>= 8)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(7)
pix1_noisy = pix1[:n_corr] + np.random.normal(0, pixel_noise_8pt, (n_corr, 2))
pix2_noisy = pix2[:n_corr] + np.random.normal(0, pixel_noise_8pt, (n_corr, 2))

# Convert to normalised coordinates
pts1_norm = (K_inv @ np.hstack([pix1_noisy, np.ones((n_corr, 1))]).T).T
pts2_norm = (K_inv @ np.hstack([pix2_noisy, np.ones((n_corr, 1))]).T).T

# Build the constraint matrix A
A_8pt = np.zeros((n_corr, 9))
for i in range(n_corr):
    x1, y1, _ = pts1_norm[i]
    x2, y2, _ = pts2_norm[i]
    A_8pt[i] = [x2*x1, x2*y1, x2, y2*x1, y2*y1, y2, x1, y1, 1]

# SVD solve
_, _, Vt_8 = np.linalg.svd(A_8pt)
E_raw = Vt_8[-1].reshape(3, 3)

# Enforce rank 2
U, S, Vt = np.linalg.svd(E_raw)
S_corrected = np.array([(S[0] + S[1]) / 2, (S[0] + S[1]) / 2, 0])
E_est = U @ np.diag(S_corrected) @ Vt

# Normalise for comparison
E_est_n = E_est / np.linalg.norm(E_est, 'fro')
E_true_n = E_true / np.linalg.norm(E_true, 'fro')
# Handle sign ambiguity
if np.sum(E_est_n * E_true_n) < 0:
    E_est_n = -E_est_n

print('True E (normalised):')
print(E_true_n.round(4))
print('\nEstimated E (normalised):')
print(E_est_n.round(4))
print(f'\nFrobenius error: {np.linalg.norm(E_est_n - E_true_n):.4f}')

In [ ]:
# Verify epipolar constraint: p2^T E p1 should be ~0 for all correspondences
residuals = []
for i in range(n_corr):
    r = abs(pts2_norm[i] @ E_est @ pts1_norm[i])
    residuals.append(r)
residuals = np.array(residuals)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(n_corr), residuals, color='steelblue', edgecolor='k', lw=0.3)
ax.axhline(residuals.mean(), color='tomato', ls='--',
           label=f'mean = {residuals.mean():.2e}')
ax.set_xlabel('correspondence index')
ax.set_ylabel(r'$|\hat{p}_2^T E \hat{p}_1|$')
ax.set_title('Epipolar constraint residuals (should be near zero)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 32.3 Triangulation

Given corresponding points in two images and the camera projection matrices, we
reconstruct the 3D position by finding the point that best satisfies both projection
equations simultaneously.

The **DLT triangulation** sets up a $4 \times 4$ homogeneous system from the two
projection matrices $P_1, P_2$ and pixel observations $(u_1, v_1)$, $(u_2, v_2)$.
The solution is the right null vector of $A$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
pixel_noise_tri = 1.0   # pixel noise std (try 0, 0.5, 2.0, 5.0)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
pix1_n = pix1 + np.random.normal(0, pixel_noise_tri, pix1.shape)
pix2_n = pix2 + np.random.normal(0, pixel_noise_tri, pix2.shape)

# Projection matrices
P1 = K @ np.hstack([R1, -R1 @ t1.reshape(3,1)])
P2 = K @ np.hstack([R2, -R2 @ t2.reshape(3,1)])

def triangulate_dlt(P1, P2, pix1, pix2):
    """DLT triangulation for multiple point pairs."""
    n = len(pix1)
    pts3d = np.zeros((n, 3))
    for i in range(n):
        u1, v1 = pix1[i]
        u2, v2 = pix2[i]
        A = np.array([
            u1 * P1[2] - P1[0],
            v1 * P1[2] - P1[1],
            u2 * P2[2] - P2[0],
            v2 * P2[2] - P2[1]
        ])
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        pts3d[i] = X[:3] / X[3]
    return pts3d

triangulated = triangulate_dlt(P1, P2, pix1_n, pix2_n)
errors = np.linalg.norm(triangulated - points_3d, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(points_3d[:, 0], points_3d[:, 2], c='steelblue', s=60, label='true 3D', zorder=5)
ax.scatter(triangulated[:, 0], triangulated[:, 2], c='tomato', s=40, marker='x', label='triangulated')
for i in range(n_points):
    ax.plot([points_3d[i,0], triangulated[i,0]],
            [points_3d[i,2], triangulated[i,2]], 'gray', alpha=0.3)
ax.plot(0, 0, 'k^', ms=12, label='Camera 1')
ax.plot(baseline, 0, 'ks', ms=12, label='Camera 2')
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_title(f'Triangulation (noise = {pixel_noise_tri} px, mean err = {errors.mean():.3f} m)',
             fontsize=12)
ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(points_3d[:, 2], errors, c='orange', s=40, edgecolors='k', lw=0.5)
ax.set_xlabel('Point depth Z (m)'); ax.set_ylabel('Reconstruction error (m)')
ax.set_title('Error vs depth: farther points have larger error', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Three diagnostic plots: error vs (a) baseline, (b) pixel noise, (c) depth

np.random.seed(0)
# (a) Baseline sweep
baselines = np.linspace(0.1, 5.0, 20)
err_vs_base = []
for b in baselines:
    t2b = np.array([b, 0, 0])
    P2b = K @ np.hstack([R2, -R2 @ t2b.reshape(3,1)])
    pix2b, _ = project(K, R2, t2b, points_3d)
    pix2b_n = pix2b + np.random.normal(0, 1.0, pix2b.shape)
    tri_b = triangulate_dlt(P1, P2b, pix1, pix2b_n)
    err_vs_base.append(np.linalg.norm(tri_b - points_3d, axis=1).mean())

# (b) Noise sweep
noises = np.linspace(0, 5.0, 20)
err_vs_noise = []
for sigma in noises:
    p1n = pix1 + np.random.normal(0, max(sigma, 1e-10), pix1.shape)
    p2n = pix2 + np.random.normal(0, max(sigma, 1e-10), pix2.shape)
    tri_n = triangulate_dlt(P1, P2, p1n, p2n)
    err_vs_noise.append(np.linalg.norm(tri_n - points_3d, axis=1).mean())

# (c) Depth sweep (fixed baseline=1, noise=1)
depth_vals = np.linspace(2, 20, 20)
err_vs_depth = []
for d in depth_vals:
    pts_d = np.column_stack([np.zeros(5), np.zeros(5), np.full(5, d)])
    p1d, _ = project(K, R1, t1, pts_d)
    p2d, _ = project(K, R2, t2, pts_d)
    p1d_n = p1d + np.random.normal(0, 1.0, p1d.shape)
    p2d_n = p2d + np.random.normal(0, 1.0, p2d.shape)
    tri_d = triangulate_dlt(P1, P2, p1d_n, p2d_n)
    err_vs_depth.append(np.linalg.norm(tri_d - pts_d, axis=1).mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(baselines, err_vs_base, 'o-', color='steelblue', ms=4)
axes[0].set_xlabel('Baseline (m)'); axes[0].set_ylabel('Mean error (m)')
axes[0].set_title('(a) Error vs baseline', fontsize=12)

axes[1].plot(noises, err_vs_noise, 's-', color='tomato', ms=4)
axes[1].set_xlabel('Pixel noise std (px)'); axes[1].set_ylabel('Mean error (m)')
axes[1].set_title('(b) Error vs pixel noise', fontsize=12)

axes[2].plot(depth_vals, err_vs_depth, '^-', color='forestgreen', ms=4)
axes[2].set_xlabel('Point depth (m)'); axes[2].set_ylabel('Mean error (m)')
axes[2].set_title('(c) Error vs depth', fontsize=12)

plt.suptitle('Triangulation accuracy depends on baseline, noise, and depth', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Key observations:**
- Larger baseline improves depth accuracy (wider triangulation angle).
- Error grows roughly linearly with pixel noise.
- Error grows quadratically with depth, because the parallax shrinks.

## 32.4 Relative Pose from the Essential Matrix

The essential matrix $E$ can be decomposed into the relative rotation $R$ and
translation direction $\hat{t}$ using SVD:

$$E = U \, \text{diag}(1, 1, 0) \, V^T$$

This yields **four** possible $(R, t)$ solutions. The correct one is identified by the
**chirality check**: all triangulated points must have positive depth in both cameras.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# We use E_est from Section 32.2
# ─────────────────────────────────────────────────────────────────────────────

U_e, S_e, Vt_e = np.linalg.svd(E_est)

# W matrix for decomposition
W = np.array([[0, -1, 0],
              [1,  0, 0],
              [0,  0, 1]], dtype=float)

# Four candidate solutions
R_candidates = [
    U_e @ W @ Vt_e,
    U_e @ W @ Vt_e,
    U_e @ W.T @ Vt_e,
    U_e @ W.T @ Vt_e
]
t_candidates = [
    U_e[:, 2],
   -U_e[:, 2],
    U_e[:, 2],
   -U_e[:, 2]
]

# Fix determinant (ensure proper rotation)
for i in range(4):
    if np.linalg.det(R_candidates[i]) < 0:
        R_candidates[i] = -R_candidates[i]
        t_candidates[i] = -t_candidates[i]

# Chirality check: triangulate a few points and count those with positive depth
best_count = -1
best_idx = 0
chiral_counts = []

for ci in range(4):
    Rc = R_candidates[ci]
    tc = t_candidates[ci]
    P2c = K @ np.hstack([Rc, -Rc @ tc.reshape(3,1)])
    tri_c = triangulate_dlt(P1, P2c, pix1_noisy[:n_corr], pix2_noisy[:n_corr])
    # Check depth in camera 1 (Z > 0)
    depth1 = tri_c[:, 2]
    # Check depth in camera 2
    pts_cam2 = (Rc @ (tri_c - tc).T).T
    depth2 = pts_cam2[:, 2]
    count = np.sum((depth1 > 0) & (depth2 > 0))
    chiral_counts.append(count)
    if count > best_count:
        best_count = count
        best_idx = ci

print('Chirality check results:')
for i in range(4):
    marker = ' <== CORRECT' if i == best_idx else ''
    print(f'  Solution {i}: {chiral_counts[i]}/{n_corr} points in front of both cameras{marker}')

In [ ]:
# Visualise all 4 candidate camera configurations
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ci, ax in enumerate(axes.flat):
    Rc = R_candidates[ci]
    tc = t_candidates[ci]
    P2c = K @ np.hstack([Rc, -Rc @ tc.reshape(3,1)])
    tri_c = triangulate_dlt(P1, P2c, pix1_noisy[:n_corr], pix2_noisy[:n_corr])

    depth1 = tri_c[:, 2]
    pts_cam2 = (Rc @ (tri_c - tc).T).T
    depth2 = pts_cam2[:, 2]
    in_front = (depth1 > 0) & (depth2 > 0)

    col = 'forestgreen' if ci == best_idx else 'tomato'
    label_tag = 'CORRECT' if ci == best_idx else 'wrong'

    ax.scatter(tri_c[in_front, 0], tri_c[in_front, 2],
              c='forestgreen', s=30, label=f'in front ({in_front.sum()})', zorder=5)
    ax.scatter(tri_c[~in_front, 0], tri_c[~in_front, 2],
              c='tomato', s=20, marker='x', label=f'behind ({(~in_front).sum()})')
    ax.plot(0, 0, 'k^', ms=10)
    ax.plot(tc[0], tc[2], 'ks', ms=10)
    # Camera 2 direction
    ax.annotate('', xy=(tc[0] + Rc[2,0]*0.5, tc[2] + Rc[2,2]*0.5),
                xytext=(tc[0], tc[2]),
                arrowprops=dict(arrowstyle='->', color=col, lw=2))
    ax.set_title(f'Solution {ci} ({label_tag})', fontsize=12, color=col, fontweight='bold')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
    ax.legend(fontsize=8)

plt.suptitle('Four E decomposition candidates (chirality test)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare recovered R, t with ground truth
R_best = R_candidates[best_idx]
t_best = t_candidates[best_idx]

# True relative: R_rel = I, t_rel direction = [1, 0, 0]
R_rel_true = np.eye(3)
t_dir_true = t_rel / np.linalg.norm(t_rel)
t_dir_est  = t_best / np.linalg.norm(t_best)

# Rotation error (angle of R_err = R_est R_true^T)
R_err = R_best @ R_rel_true.T
cos_angle = np.clip((np.trace(R_err) - 1) / 2, -1, 1)
rot_err_deg = np.degrees(np.arccos(cos_angle))

# Translation direction error
if np.dot(t_dir_est, t_dir_true) < 0:
    t_dir_est = -t_dir_est  # sign ambiguity
t_angle_err = np.degrees(np.arccos(np.clip(np.dot(t_dir_est, t_dir_true), -1, 1)))

print(f'Rotation error:              {rot_err_deg:.3f} deg')
print(f'Translation direction error:  {t_angle_err:.3f} deg')
print(f'\nRecovered t direction: {t_dir_est.round(4)}')
print(f'True t direction:      {t_dir_true.round(4)}')

## Capstone: Full Two View Reconstruction Pipeline

We now assemble the complete pipeline:
1. Generate a 3D scene with 30 points.
2. Project to two cameras with known poses.
3. Add pixel noise.
4. Estimate $E$ with the 8 point algorithm.
5. Decompose $E$ into $R, t$ with chirality check.
6. Triangulate all points.
7. Compare reconstructed 3D with ground truth.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(2024)
n_cap       = 30
cap_noise   = 1.0        # pixel noise
cap_base    = 1.5        # baseline
cap_yaw_deg = 5.0        # camera 2 slight rotation
# ─────────────────────────────────────────────────────────────────────────────

# Scene
scene = np.column_stack([
    np.random.uniform(-3, 3, n_cap),
    np.random.uniform(-2, 2, n_cap),
    np.random.uniform(4, 10, n_cap)
])

# Camera poses
R1c = np.eye(3); t1c = np.zeros(3)
yaw_r = np.radians(cap_yaw_deg)
R2c = np.array([[ np.cos(yaw_r), 0, np.sin(yaw_r)],
                [ 0,             1, 0            ],
                [-np.sin(yaw_r), 0, np.cos(yaw_r)]])
t2c = np.array([cap_base, 0.0, 0.0])

# Project
pA, _ = project(K, R1c, t1c, scene)
pB, _ = project(K, R2c, t2c, scene)
pA_n = pA + np.random.normal(0, cap_noise, pA.shape)
pB_n = pB + np.random.normal(0, cap_noise, pB.shape)

# Step 1: Estimate E
nA = (K_inv @ np.hstack([pA_n, np.ones((n_cap,1))]).T).T
nB = (K_inv @ np.hstack([pB_n, np.ones((n_cap,1))]).T).T

A_cap = np.zeros((n_cap, 9))
for i in range(n_cap):
    x1, y1, _ = nA[i]
    x2, y2, _ = nB[i]
    A_cap[i] = [x2*x1, x2*y1, x2, y2*x1, y2*y1, y2, x1, y1, 1]
_, _, Vt_c = np.linalg.svd(A_cap)
Ec_raw = Vt_c[-1].reshape(3,3)
Uc, Sc, Vtc = np.linalg.svd(Ec_raw)
Sc_fix = np.array([(Sc[0]+Sc[1])/2, (Sc[0]+Sc[1])/2, 0])
Ec = Uc @ np.diag(Sc_fix) @ Vtc

# Step 2: Decompose E
Ud, Sd, Vtd = np.linalg.svd(Ec)
R_cands = [Ud @ W @ Vtd, Ud @ W @ Vtd, Ud @ W.T @ Vtd, Ud @ W.T @ Vtd]
t_cands = [Ud[:,2], -Ud[:,2], Ud[:,2], -Ud[:,2]]
for i in range(4):
    if np.linalg.det(R_cands[i]) < 0:
        R_cands[i] = -R_cands[i]
        t_cands[i] = -t_cands[i]

# Chirality
PA = K @ np.hstack([R1c, np.zeros((3,1))])
best_n = -1; best_i = 0
for ci in range(4):
    PBc = K @ np.hstack([R_cands[ci], -R_cands[ci] @ t_cands[ci].reshape(3,1)])
    tri_test = triangulate_dlt(PA, PBc, pA_n, pB_n)
    d1 = tri_test[:, 2]
    d2 = (R_cands[ci] @ (tri_test - t_cands[ci]).T).T[:, 2]
    cnt = np.sum((d1 > 0) & (d2 > 0))
    if cnt > best_n:
        best_n = cnt; best_i = ci

R_rec = R_cands[best_i]
t_rec = t_cands[best_i]

# Step 3: Triangulate with recovered pose
PB_rec = K @ np.hstack([R_rec, -R_rec @ t_rec.reshape(3,1)])
recon = triangulate_dlt(PA, PB_rec, pA_n, pB_n)

# Scale ambiguity: align scale using median depth ratio
scale = np.median(scene[:, 2]) / np.median(recon[:, 2])
recon_scaled = recon * scale

recon_err = np.linalg.norm(recon_scaled - scene, axis=1)

# Rotation and translation error
R_rel_true_cap = R2c @ R1c.T
R_rel_err = R_rec @ R_rel_true_cap.T
rot_err_cap = np.degrees(np.arccos(np.clip((np.trace(R_rel_err)-1)/2, -1, 1)))

t_dir_true_cap = (t2c - t1c); t_dir_true_cap = t_dir_true_cap / np.linalg.norm(t_dir_true_cap)
t_dir_rec = t_rec / np.linalg.norm(t_rec)
if np.dot(t_dir_rec, t_dir_true_cap) < 0:
    t_dir_rec = -t_dir_rec
t_err_cap = np.degrees(np.arccos(np.clip(np.dot(t_dir_rec, t_dir_true_cap), -1, 1)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(scene[:, 0], scene[:, 2], c='steelblue', s=50, label='ground truth', zorder=5)
ax.scatter(recon_scaled[:, 0], recon_scaled[:, 2], c='tomato', s=30, marker='x', label='reconstructed')
for i in range(n_cap):
    ax.plot([scene[i,0], recon_scaled[i,0]],
            [scene[i,2], recon_scaled[i,2]], 'gray', alpha=0.3)
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_title('3D reconstruction (top view)', fontsize=12)
ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(scene[:, 0], scene[:, 1], c='steelblue', s=50, label='ground truth', zorder=5)
ax.scatter(recon_scaled[:, 0], recon_scaled[:, 1], c='tomato', s=30, marker='x', label='reconstructed')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('3D reconstruction (front view)', fontsize=12)
ax.legend(fontsize=9)

ax = axes[2]
ax.bar(range(n_cap), recon_err, color='orange', edgecolor='k', lw=0.3)
ax.axhline(recon_err.mean(), color='tomato', ls='--',
           label=f'mean = {recon_err.mean():.3f} m')
ax.set_xlabel('point index'); ax.set_ylabel('error (m)')
ax.set_title('Per point reconstruction error', fontsize=12)
ax.legend()

plt.tight_layout()
plt.show()

print('=== Capstone Report ===')
print(f'Rotation error:              {rot_err_cap:.3f} deg')
print(f'Translation direction error:  {t_err_cap:.3f} deg')
print(f'Reconstruction RMSE:          {np.sqrt((recon_err**2).mean()):.4f} m')
print(f'Reconstruction mean error:    {recon_err.mean():.4f} m')

**Capstone takeaways:**
- The 8 point algorithm, combined with chirality checking, recovers the relative pose well.
- Scale is inherently ambiguous in monocular reconstruction; we resolved it using the ground truth median depth.
- With 1 pixel noise and a 1.5 m baseline, reconstruction RMSE is typically below 0.5 m for points at 4 to 10 m depth.

---

## Exercises

### Exercise 32.1: Baseline vs accuracy

Vary the baseline from 0.1 m to 5 m in 20 steps. For each, generate 30 points at
depth 5 m, project with 1 px noise, triangulate, and record the mean error.
Plot mean reconstruction error vs baseline. At what baseline does the error flatten out?

In [ ]:
# Your code here

### Exercise 32.2: 8 point algorithm robustness

Run the 8 point algorithm 50 times with different random noise realisations
(pixel noise $\sigma = 2$). Record the Frobenius error of $E$ each time.
Plot a histogram of the errors. What is the median error?

In [ ]:
# Your code here

### Exercise 32.3: Epipolar line visualisation

Choose a point at the image centre of camera 1. Compute its epipolar line in camera 2
for three different baselines: 0.5, 1.0, 2.0 m. Overlay all three lines on the same
image. How does the line slope change with baseline direction?

In [ ]:
# Your code here

### Exercise 32.4: Minimum correspondences

The 8 point algorithm needs at least 8 points. Run it with $N = 8, 10, 15, 20, 30, 50$
correspondences (pixel noise = 1.0). Plot the $E$ estimation error vs $N$.
How many correspondences do you need for reliable estimation?

In [ ]:
# Your code here

### Exercise 32.5: Forward motion degeneracy (challenge)

Place camera 2 directly in front of camera 1 (pure forward motion: $t = [0, 0, 1]$).
Generate points and run the full pipeline. What happens to the epipolar lines?
Why does the 8 point algorithm struggle in this configuration?

In [ ]:
# Your code here